Step 1:Environment Setup

In [23]:

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers pypdf

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 10.9 MB/s eta 0:00:00


Step 2:Load Unsloth Dynamic 4-Bit Model

In [24]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Ll

Step 3: Memory Optimization & VRAM Verification

In [25]:
import torch

gpu_stats = torch.cuda.get_device_properties(0)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
reserved_memory = round(torch.cuda.memory_reserved(0) / 1024 / 1024 / 1024, 3)

print(f"GPU Model: {gpu_stats.name}")
print(f"Total GPU VRAM: {max_memory} GB")
print(f"VRAM Currently Allocated: {reserved_memory} GB")

GPU Model: Tesla T4
Total GPU VRAM: 14.563 GB
VRAM Currently Allocated: 8.797 GB


Step 4: Document Chunking & FAISS Vector Indexing

In [30]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Domain-specific medical knowledge
raw_text = """
Diabetes mellitus is a chronic metabolic disorder characterized by elevated
blood glucose levels. Type 1 diabetes is generally caused by autoimmune
destruction of insulin-producing beta cells. Type 2 diabetes is commonly
associated with insulin resistance and impaired insulin secretion.

Hypertension is a condition in which blood pressure remains consistently
elevated. Long-term uncontrolled hypertension may increase the risk of
cardiovascular disease, stroke, and kidney problems. Regular blood pressure
monitoring is important.

Anemia is a condition in which the blood does not contain enough healthy red
blood cells or hemoglobin to carry sufficient oxygen to body tissues.
Iron deficiency is one common cause of anemia. Common symptoms can include
fatigue and weakness.

Asthma is a chronic respiratory condition involving inflammation and
narrowing of the airways. Common symptoms include coughing, wheezing,
chest tightness, and difficulty breathing.

Vitamin D is important for bone health and calcium regulation. Vitamin D
deficiency can contribute to weakened bones. Sources of vitamin D include
sunlight and certain foods.
"""

# Split documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

docs = [
    Document(page_content=chunk)
    for chunk in text_splitter.split_text(raw_text)
]

print("Number of document chunks:", len(docs))

# Create embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create FAISS vector database
vector_db = FAISS.from_documents(
    docs,
    embedding_model
)

# Create retriever
retriever = vector_db.as_retriever(
    search_kwargs={"k": 2}
)

print("Documents successfully indexed in FAISS vector store!")
print("Retriever is ready.")

Number of document chunks: 5


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Documents successfully indexed in FAISS vector store!
Retriever is ready.


Step 5: PDF Document Loader (Dynamic Processing Option)

In [32]:
from langchain_community.document_loaders import PyPDFLoader

def load_pdf_to_vectorstore(pdf_path: str):

    # Load PDF
    loader = PyPDFLoader(pdf_path)

    # Split PDF into chunks
    pdf_docs = loader.load_and_split(text_splitter)

    # Create FAISS vector database
    pdf_vector_db = FAISS.from_documents(
        pdf_docs,
        embedding_model
    )

    # Create retriever
    pdf_retriever = pdf_vector_db.as_retriever(
        search_kwargs={"k": 2}
    )

    return pdf_retriever


print("PDF Loader function is ready for custom medical documents.")

PDF Loader function is ready for custom medical documents.


Step 6: RAG Pipeline

In [35]:
# Step 6: RAG Pipeline Execution

def answer_rag_query(query: str):

    # Retrieve relevant documents
    retrieved_docs = retriever.invoke(query)

    # Combine retrieved chunks into context
    context = "\n\n".join(
        [doc.page_content for doc in retrieved_docs]
    )

    # Create grounded prompt
    messages = [
        {
            "role": "system",
            "content": (
                "You are a medical information assistant. "
                "Answer the question ONLY using the provided context. "
                "Do not invent information. "
                "If the answer is not available in the context, say: "
                "'I cannot find the answer in the provided document.'"
            )
        },
        {
            "role": "user",
            "content": (
                f"Context:\n{context}\n\n"
                f"Question: {query}"
            )
        }
    ]

    # Convert messages into model prompt
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize and move inputs to GPU
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    # Generate response
    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            use_cache=True,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only newly generated tokens
    generated_tokens = outputs[0][inputs.input_ids.shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response, context

Cell 7: In-Context RAG Test

In [36]:
user_query = "What are common symptoms of asthma?"

response, context = answer_rag_query(user_query)

print("=== USER QUERY ===")
print(user_query)

print("\n=== RETRIEVED CONTEXT ===")
print(context)

print("\n=== GENERATED RESPONSE ===")
print(response)

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== USER QUERY ===
What are common symptoms of asthma?

=== RETRIEVED CONTEXT ===
Asthma is a chronic respiratory condition involving inflammation and
narrowing of the airways. Common symptoms include coughing, wheezing,
chest tightness, and difficulty breathing.

Anemia is a condition in which the blood does not contain enough healthy red
blood cells or hemoglobin to carry sufficient oxygen to body tissues.
Iron deficiency is one common cause of anemia. Common symptoms can include
fatigue and weakness.

=== GENERATED RESPONSE ===
According to the provided context, common symptoms of asthma include:

1. Coughing
2. Wheezing
3. Chest tightness
4. Difficulty breathing


Step 8: Show Retrieved Documents

In [37]:
query = "What is diabetes mellitus?"

retrieved_docs = retriever.invoke(query)

print("=== QUERY ===")
print(query)

print("\n=== RETRIEVED DOCUMENTS ===")

for i, doc in enumerate(retrieved_docs):

    print(f"\n--- Retrieved Chunk {i + 1} ---")
    print(doc.page_content)

=== QUERY ===
What is diabetes mellitus?

=== RETRIEVED DOCUMENTS ===

--- Retrieved Chunk 1 ---
Diabetes mellitus is a chronic metabolic disorder characterized by elevated
blood glucose levels. Type 1 diabetes is generally caused by autoimmune
destruction of insulin-producing beta cells. Type 2 diabetes is commonly
associated with insulin resistance and impaired insulin secretion.

--- Retrieved Chunk 2 ---
Anemia is a condition in which the blood does not contain enough healthy red
blood cells or hemoglobin to carry sufficient oxygen to body tissues.
Iron deficiency is one common cause of anemia. Common symptoms can include
fatigue and weakness.


Step 9: Out-of-Context / Anti-Hallucination Test

In [38]:
out_of_context_query = "What is the capital of France?"

response, context = answer_rag_query(
    out_of_context_query
)

print("=== OUT-OF-CONTEXT TEST ===")

print("\nQuery:")
print(out_of_context_query)

print("\nRetrieved Context:")
print(context)

print("\nGenerated Response:")
print(response)

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== OUT-OF-CONTEXT TEST ===

Query:
What is the capital of France?

Retrieved Context:
Vitamin D is important for bone health and calcium regulation. Vitamin D
deficiency can contribute to weakened bones. Sources of vitamin D include
sunlight and certain foods.

Asthma is a chronic respiratory condition involving inflammation and
narrowing of the airways. Common symptoms include coughing, wheezing,
chest tightness, and difficulty breathing.

Generated Response:
I cannot find the answer in the provided document.


Step 10: Multiple Questions Test

In [39]:
# Step 10: Multiple RAG Test Queries

test_questions = [
    "What is diabetes mellitus?",
    "What are common symptoms of asthma?",
    "What is hypertension?",
    "What is one common cause of anemia?",
    "Why is vitamin D important?"
]

for question in test_questions:

    response, context = answer_rag_query(question)

    print("=" * 70)
    print("QUESTION:")
    print(question)

    print("\nRETRIEVED CONTEXT:")
    print(context)

    print("\nGENERATED ANSWER:")
    print(response)

    print("=" * 70)

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
What is diabetes mellitus?

RETRIEVED CONTEXT:
Diabetes mellitus is a chronic metabolic disorder characterized by elevated
blood glucose levels. Type 1 diabetes is generally caused by autoimmune
destruction of insulin-producing beta cells. Type 2 diabetes is commonly
associated with insulin resistance and impaired insulin secretion.

Anemia is a condition in which the blood does not contain enough healthy red
blood cells or hemoglobin to carry sufficient oxygen to body tissues.
Iron deficiency is one common cause of anemia. Common symptoms can include
fatigue and weakness.

GENERATED ANSWER:
Diabetes mellitus is a chronic metabolic disorder characterized by elevated blood glucose levels.


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
What are common symptoms of asthma?

RETRIEVED CONTEXT:
Asthma is a chronic respiratory condition involving inflammation and
narrowing of the airways. Common symptoms include coughing, wheezing,
chest tightness, and difficulty breathing.

Anemia is a condition in which the blood does not contain enough healthy red
blood cells or hemoglobin to carry sufficient oxygen to body tissues.
Iron deficiency is one common cause of anemia. Common symptoms can include
fatigue and weakness.

GENERATED ANSWER:
According to the provided context, common symptoms of asthma include:

1. Coughing
2. Wheezing
3. Chest tightness
4. Difficulty breathing


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
What is hypertension?

RETRIEVED CONTEXT:
Hypertension is a condition in which blood pressure remains consistently
elevated. Long-term uncontrolled hypertension may increase the risk of
cardiovascular disease, stroke, and kidney problems. Regular blood pressure
monitoring is important.

Diabetes mellitus is a chronic metabolic disorder characterized by elevated
blood glucose levels. Type 1 diabetes is generally caused by autoimmune
destruction of insulin-producing beta cells. Type 2 diabetes is commonly
associated with insulin resistance and impaired insulin secretion.

GENERATED ANSWER:
Hypertension is a condition in which blood pressure remains consistently elevated.


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
What is one common cause of anemia?

RETRIEVED CONTEXT:
Anemia is a condition in which the blood does not contain enough healthy red
blood cells or hemoglobin to carry sufficient oxygen to body tissues.
Iron deficiency is one common cause of anemia. Common symptoms can include
fatigue and weakness.

Diabetes mellitus is a chronic metabolic disorder characterized by elevated
blood glucose levels. Type 1 diabetes is generally caused by autoimmune
destruction of insulin-producing beta cells. Type 2 diabetes is commonly
associated with insulin resistance and impaired insulin secretion.

GENERATED ANSWER:
One common cause of anemia is iron deficiency.
QUESTION:
Why is vitamin D important?

RETRIEVED CONTEXT:
Vitamin D is important for bone health and calcium regulation. Vitamin D
deficiency can contribute to weakened bones. Sources of vitamin D include
sunlight and certain foods.

Hypertension is a condition in which blood pressure remains consistently
elevated. Long-term uncontro

Step:11 VRAM Verification After RAG

In [40]:
# Step 11: VRAM Usage Verification

if torch.cuda.is_available():

    allocated_memory = (
        torch.cuda.memory_allocated(0) / 1024**3
    )

    reserved_memory = (
        torch.cuda.memory_reserved(0) / 1024**3
    )

    total_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3
    )

    print("=== VRAM USAGE ===")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {total_memory:.2f} GB")
    print(f"Allocated VRAM: {allocated_memory:.2f} GB")
    print(f"Reserved VRAM: {reserved_memory:.2f} GB")

else:

    print("CUDA GPU is not available.")

=== VRAM USAGE ===
GPU: Tesla T4
Total VRAM: 14.56 GB
Allocated VRAM: 9.04 GB
Reserved VRAM: 9.21 GB


Step 12 : Memory Optimization

In [41]:
# Step 12: Memory Optimization

print("=== MEMORY OPTIMIZATION SETTINGS ===")
print("4-bit quantization: ENABLED")
print("Inference mode: ENABLED")
print("Maximum sequence length:", max_seq_length)
print("KV cache: ENABLED")

torch.cuda.empty_cache()

print("\nCUDA cache cleared successfully.")

=== MEMORY OPTIMIZATION SETTINGS ===
4-bit quantization: ENABLED
Inference mode: ENABLED
Maximum sequence length: 2048
KV cache: ENABLED

CUDA cache cleared successfully.


Cell 13 — Final Complete Demonstration

In [42]:
query = "What are the common symptoms of asthma?"

response, context = answer_rag_query(query)

print("=" * 70)
print("        RETRIEVAL-AUGMENTED GENERATION SYSTEM")
print("=" * 70)

print("\nUSER QUERY:")
print(query)

print("\nRETRIEVED INFORMATION:")
print(context)

print("\nGENERATED GROUNDED RESPONSE:")
print(response)

print("\nMODEL:")
print("Unsloth Llama-3.2-3B-Instruct 4-bit")

print("\nVECTOR DATABASE:")
print("FAISS")

print("\nEMBEDDING MODEL:")
print("all-MiniLM-L6-v2")

print("\nQUANTIZATION:")
print("4-bit")

print("=" * 70)

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


        RETRIEVAL-AUGMENTED GENERATION SYSTEM

USER QUERY:
What are the common symptoms of asthma?

RETRIEVED INFORMATION:
Asthma is a chronic respiratory condition involving inflammation and
narrowing of the airways. Common symptoms include coughing, wheezing,
chest tightness, and difficulty breathing.

Anemia is a condition in which the blood does not contain enough healthy red
blood cells or hemoglobin to carry sufficient oxygen to body tissues.
Iron deficiency is one common cause of anemia. Common symptoms can include
fatigue and weakness.

GENERATED GROUNDED RESPONSE:
According to the provided context, the common symptoms of asthma are:

1. Coughing
2. Wheezing
3. Chest tightness
4. Difficulty breathing

MODEL:
Unsloth Llama-3.2-3B-Instruct 4-bit

VECTOR DATABASE:
FAISS

EMBEDDING MODEL:
all-MiniLM-L6-v2

QUANTIZATION:
4-bit
